In [1]:
# Import libraries

from pathlib import Path
import os
from dotenv import load_dotenv

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()


True

In [2]:
# Load OpenAI API key from environment variable

openai_api_key = os.getenv("OPENAI_API_KEY")
print("Key Loaded:", openai_api_key is not None)

Key Loaded: True


In [3]:
# Load policies from the "policies" directory

policies = Path("policies")
files = sorted(policies.glob("**/*.md"))
print(len(files), "files found in policies directory.")

for f in files:
    print(f.name)

10 files found in policies directory.
01_annual_leave_policy.md
02_sick_leave_policy.md
03_remote_and_hybrid_work_policy.md
04_business_travel_policy.md
05_expense_reimbursement_policy.md
06_it_equipment_and_security_policy.md
07_learning_and_development_policy.md
08_parental_leave_policy.md
09_code_of_conduct_gifts_policy.md
10_working_hours_and_overtime_policy.md


In [4]:
docs = {f.name: f.read_text(encoding="utf-8") for f in files}

print(docs["01_annual_leave_policy.md"])

> FICTIONAL DOCUMENT — created for a recruitment assessment. Kestrel Nova Ltd is not a real company.

# Annual Leave Policy
**Kestrel Nova Ltd · Policy HR-01 · Version 3.2 · Effective 1 January 2026 · Owner: People & Culture**

## 1. Purpose
This policy explains how much paid annual leave employees receive and how to request it.

## 2. Eligibility
All permanent employees are entitled to annual leave. Employees on probation (the first **3 months** of employment) earn leave but may only take it with written approval from their line manager.

## 3. Entitlement
- Full-time employees receive **20 working days** of paid annual leave per calendar year after completing probation.
- Employees with more than 5 years of continuous service receive **25 working days**.
- Part-time employees receive leave in proportion to their contracted hours.
- Public holidays are in addition to annual leave.

## 4. Requesting leave
- All leave must be requested through the PeopleHub portal.
- Requests for more t

In [5]:
# Split the text into sections based on headers

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "policy"), ("##", "section")]
)

sample = splitter.split_text(docs["01_annual_leave_policy.md"])
for c in sample:
    print(c.metadata)
    print(c.page_content[:100])
    print("---")

{}
> FICTIONAL DOCUMENT — created for a recruitment assessment. Kestrel Nova Ltd is not a real company.
---
{'policy': 'Annual Leave Policy'}
**Kestrel Nova Ltd · Policy HR-01 · Version 3.2 · Effective 1 January 2026 · Owner: People & Culture
---
{'policy': 'Annual Leave Policy', 'section': '1. Purpose'}
This policy explains how much paid annual leave employees receive and how to request it.
---
{'policy': 'Annual Leave Policy', 'section': '2. Eligibility'}
All permanent employees are entitled to annual leave. Employees on probation (the first **3 months**
---
{'policy': 'Annual Leave Policy', 'section': '3. Entitlement'}
- Full-time employees receive **20 working days** of paid annual leave per calendar year after compl
---
{'policy': 'Annual Leave Policy', 'section': '4. Requesting leave'}
- All leave must be requested through the PeopleHub portal.
- Requests for more than 3 consecutive w
---
{'policy': 'Annual Leave Policy', 'section': '5. Carrying over leave'}
- Up to **5 unused da

In [6]:
# Split the text into sections based on headers and store in a list of documents

import re

chunks = []
for f in files:
    text = f.read_text(encoding="utf-8")
    text = re.sub(r"^>.*FICTIONAL.*\n", "", text)          # drop disclaimer
    meta = re.search(r"Policy (\S+) · Version (\S+)", text)
    for chunk in splitter.split_text(text):
        if "section" not in chunk.metadata:                  # skip title/metadata-only chunk
            continue
        chunk.metadata.update(
            source=f.name,
            policy_id=meta.group(1) if meta else None,
            version=meta.group(2) if meta else None,
            chunk_id=f"{f.name}#{chunk.metadata['section']}",
        )
        chunk.page_content = f"{chunk.metadata['policy']} > {chunk.metadata['section']}\n\n{chunk.page_content}"
        chunks.append(chunk)

print(len(chunks), "chunks")
chunks[:10]

48 chunks


[Document(metadata={'policy': 'Annual Leave Policy', 'section': '1. Purpose', 'source': '01_annual_leave_policy.md', 'policy_id': 'HR-01', 'version': '3.2', 'chunk_id': '01_annual_leave_policy.md#1. Purpose'}, page_content='Annual Leave Policy > 1. Purpose\n\nThis policy explains how much paid annual leave employees receive and how to request it.'),
 Document(metadata={'policy': 'Annual Leave Policy', 'section': '2. Eligibility', 'source': '01_annual_leave_policy.md', 'policy_id': 'HR-01', 'version': '3.2', 'chunk_id': '01_annual_leave_policy.md#2. Eligibility'}, page_content='Annual Leave Policy > 2. Eligibility\n\nAll permanent employees are entitled to annual leave. Employees on probation (the first **3 months** of employment) earn leave but may only take it with written approval from their line manager.'),
 Document(metadata={'policy': 'Annual Leave Policy', 'section': '3. Entitlement', 'source': '01_annual_leave_policy.md', 'policy_id': 'HR-01', 'version': '3.2', 'chunk_id': '01_a

In [11]:
# Create an in-memory vector store from the chunks

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)


In [8]:
print("Vectorstore created with", len(vectorstore.store), "vectors from", len(chunks), "chunks.")


Vectorstore created with 48 vectors from 48 chunks.


In [9]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [12]:
# Query the vector store for relevant chunks

question = "How many leave days can I carry over to next year?"

results = vectorstore.similarity_search_with_score(question, k=4)
for doc, score in results:
    print(round(score, 3), doc.metadata["chunk_id"])

0.723 01_annual_leave_policy.md#5. Carrying over leave
0.545 01_annual_leave_policy.md#3. Entitlement
0.526 01_annual_leave_policy.md#6. Leaving the company
0.522 02_sick_leave_policy.md#1. Entitlement


In [13]:
# Answer format 

class PolicyAnswer(BaseModel):
    answered: bool = Field(description="True only if the sources contain the answer")
    answer: str = Field(description="The answer, naming the policy it comes from. If not answered, say the policies do not cover this.")
    citations: list[str] = Field(description="The source labels in [brackets] that you used, copied exactly. Empty if not answered.")


# Label the sources for citation in the answer
def label(d):
    return f"{d.metadata['policy_id']} | {d.metadata['section']}"

def format_context(docs):
    return "\n\n".join(f"[{label(d)}]\n{d.page_content}" for d in docs)

In [ ]:
# System prompt for the LLM to answer questions about the policies

SYSTEM_PROMPT = """You are an HR assistant for Kestrel Nova Ltd that answers employee questions about company policies.

Rules:
- Answer using ONLY the policy context provided. Do not use outside knowledge.
- If the sources do not contain the answer, set answered to false and say the policies do not cover it. Do not guess.
- If the sources only partly answer the question, answer the part they cover and say what is not covered.
- Whenever answered is true, you must list in the citations field every source label (shown in [brackets]) you used, copied exactly.

Style:
- Write in plain, conversational English, as if speaking to a colleague. Address the employee as "you".
- Explain the answer in your own words; do not copy sentences from the policy.
- Include any conditions, deadlines or exceptions that affect the answer.
- Use plain text only: no Markdown, no asterisks, no bullet points.
- Do not put source labels in the answer text; list them only in the citations field.
"""



In [36]:
# Promt template 

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Question: {question}\n\nSources:\n{context}")
])

In [37]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | llm.with_structured_output(PolicyAnswer)

In [46]:
q = "Which days am I expected to be in the office?"

results = vectorstore.similarity_search_with_score(q, k=4)
for d, s in results:
    print(round(s, 3), label(d))

docs = [d for d, _ in results]
raw = chain.invoke({"context": format_context(docs), "question": q})
print(raw)

0.53 HR-05 | 1. Hybrid working model
0.497 HR-03 | 2. Core hours
0.436 HR-02 | 2. Reporting absence
0.432 HR-01 | 4. Requesting leave
answered=True answer="As a hybrid employee, you are expected to be in the Lagos office at least 2 days per week, specifically on Tuesday and Thursday, which are designated as anchor days. On other days, you can work from home if you have your line manager's agreement." citations=['HR-05 | 1. Hybrid working model']


In [42]:
FALLBACK = "The policy documents provided do not contain an answer to this question."

def ask(question, k=4, min_score=0.4):
    results = vectorstore.similarity_search_with_score(question, k=k)

    # Guard 1: nothing relevant retrieved, so don't call the LLM at all
    if not results or results[0][1] < min_score:
        return {"answered": False, "answer": FALLBACK, "citations": []}

    docs = [doc for doc, _ in results]
    result = chain.invoke({
        "context": format_context(docs),
        "question": question,
    })

    # The LLM itself says it's not covered: use its wording
    if not result.answered:
        return {"answered": False, "answer": result.answer, "citations": []}

    # Guard 2: it claims an answer but cites nothing real, so treat the answer as unsafe
    retrieved_labels = {label(d) for d in docs}
    valid = [c for c in result.citations if c in retrieved_labels]
    if not valid:
        return {"answered": False, "answer": FALLBACK, "citations": []}

    return {"answered": True, "answer": result.answer, "citations": valid}

In [43]:
r = ask("How many leave days can I carry over to next year?")
print(r["answer"])
print(r["answered"])
print(r["citations"])


According to the Annual Leave Policy, you can carry over up to 5 unused leave days into the next calendar year. However, these carried-over days must be used by 31 March of the following year, or they will be lost. Any days beyond the 5-day limit will also be lost at the end of the year unless you get written approval from the Head of People for an exception.
True
['HR-01 | 5. Carrying over leave']


In [44]:
for q in [
    "What is the company dress code?",              # should refuse
    "Can I claim taxi fares on a business trip?",   # should answer from Travel + Expenses
]:
    r = ask(q)
    print("Q:", q)
    print(r["answer"])
    print(r["answered"], r["citations"])
    print("---")

Q: What is the company dress code?
The policy documents provided do not contain an answer to this question.
False []
---
Q: Can I claim taxi fares on a business trip?
Yes, you can claim taxi fares on a business trip. Since taxi fares are not booked through the Travel Desk, you would need to submit these claims under the Expense Reimbursement Policy. Just remember to submit your claims within 30 days of the expense and include a receipt for the taxi fare. If you don't have a receipt and the fare is below NGN 5,000, you can still claim it with a short explanation.
True ['FIN-03 | 6. Claims', 'FIN-01 | 2. Submitting claims']
---


In [41]:
for q in [
    "How many leave days can I carry over to next year?",
    "Can I claim taxi fares on a business trip?",
    "What is the company dress code?",
    "Do we get a pension?",
    "What time does the canteen open?",
]:
    top = vectorstore.similarity_search_with_score(q, k=1)[0][1]
    print(round(top, 3), q)

0.723 How many leave days can I carry over to next year?
0.613 Can I claim taxi fares on a business trip?
0.363 What is the company dress code?
0.313 Do we get a pension?
0.314 What time does the canteen open?


In [45]:
for q in [
    "How many bereavement leave days do I get?",          # near miss: should refuse (LLM)
    "Can I expense a glass of wine at a client dinner?",  # should say no: alcohol isn't reimbursable
    "Who approves an expense of NGN 60,000?",             # line manager AND Finance
    "I'm going to be off sick for 3 days, what do I need?",  # medical certificate (more than 2 days)
    "Is business class allowed on long flights?",         # only Director level and above, flights over 6 hours
]:
    r = ask(q)
    print("Q:", q)
    print(r["answer"])
    print(r["answered"], r["citations"])
    print("---")

Q: How many bereavement leave days do I get?
The policies do not cover this.
False []
---
Q: Can I expense a glass of wine at a client dinner?
According to the Expense Reimbursement Policy, alcohol is not reimbursable, so you cannot expense a glass of wine at a client dinner. You can expense the meal itself, but not any alcoholic beverages.
True ['FIN-01 | 5. Limits and exclusions']
---
Q: Who approves an expense of NGN 60,000?
For an expense of NGN 60,000, both your line manager and the Finance department need to approve it, as it exceeds the NGN 50,000 limit for approvals.
True ['FIN-01 | 3. Approvals']
---
Q: I'm going to be off sick for 3 days, what do I need?
Since you're going to be off sick for 3 days, you need to notify your line manager by phone or chat before 10:00 on the first day of your absence. Additionally, because your absence is more than 2 consecutive working days, you will need to provide a medical certificate from a registered medical practitioner. Finally, make sur